In [1]:
%load_ext autoreload
%autoreload 2

In [10]:
import math
import os
import os.path
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import plotnine as pn
import scipy
import pycwt

In [7]:
shoreshop_path = os.path.expanduser('~/abmurraylab/shoreshop3')
input_coastsat_path = os.path.join(shoreshop_path, 'inputdata/NC_CoastSat_smoothed.zip')

_dfs = []

with zipfile.ZipFile(input_coastsat_path, 'r') as zip:
    for info in zip.infolist():
        if not info.filename.startswith('NC_CoastSat_smoothed/usa_NC_'):
            continue
        transect_id = info.filename.removeprefix('NC_CoastSat_smoothed/usa_NC_').removesuffix('.csv')

        with zip.open(info.filename) as file:
            _df = pd.read_csv(file, names=('date', 'x'))
            _df['date'] = pd.to_datetime(_df['date'])
            _df['transect'] = transect_id
            _dfs.append(_df)

input_coastsat = pd.concat(_dfs)
input_coastsat

,date,x,transect
0,1984-03-24 15:13:04,73.9,0001_0001
1,1984-05-07 15:15:23,73.8,0001_0001
2,1984-06-08 15:16:13,73.7,0001_0001
3,1984-06-24 15:16:28,70.1,0001_0001
4,1984-08-27 15:18:04,66.4,0001_0001
...,...,...,...
544,2025-10-24 15:41:17,40.4,0049_0230
545,2025-11-01 15:41:22,40.6,0049_0230
546,2025-11-25 15:41:22,42.6,0049_0230
547,2025-12-11 15:41:20,41.0,0049_0230


In [12]:
submissions_path = os.path.join(shoreshop_path, 'submissions/UserSubmissionsCleaned')

nc_cur_paths = {
    'BRIE': os.path.join(submissions_path, 'BRIE/mipNC_1980-2023-BRIE.csv'),
    'CCOST': os.path.join(submissions_path, 'CCOST/mipNC_1980-2023_DESM-reconstruction-CCOST.csv'),
    'CEERD-CHL-Devoe': os.path.join(submissions_path, 'CEERD-CHL-Devoe/mipNC_1980-2023_waves_simplified_ShoreFor_BruunSLR_CUDEMslope_ERDCCHL_SRD.csv'),
    'COCollective': os.path.join(submissions_path, 'COCollective/mipNC_1980-2023_CenturyHindcast-COCollective.csv'),
    'CoSMoS-COAST-conv_IA': os.path.join(submissions_path, 'CoSMoS-COAST_SV/NC/CoSMoS_COAST_conv_model_with_interannual/mipNC_1980-2023_CoSMoS_COAST_conv_model_with_interannual_SV.csv'),
    'CoSMoS-COAST-conv': os.path.join(submissions_path, 'CoSMoS-COAST_SV/NC/CoSMoS_COAST_conv_model_without_interannual/mipNC_1980-2023_CoSMoS_COAST_conv_model_without_interannual_SV.csv'),
    'CoSMoS-COAST-Vitousek_SciReports_2024': os.path.join(submissions_path, 'CoSMoS-COAST_SV/NC/CoSMoS_COAST_Vitousek_SciReports_2024_model/mipNC_1980-2023_CoSMoS_COAST_Vitousek_SciReports_2024_model_SV.csv'),
    'CoSMoS-COAST-CoastSat_trends': os.path.join(submissions_path, 'CoSMoS-COAST_SV/NC/CoastSat_trends/mipNC_1980-2023_CoastSat_trends_SV.csv'),
    'CoSMoS-COAST-CoastSat_trends_bruun_ssp585': os.path.join(submissions_path, 'CoSMoS-COAST_SV/NC/CoastSat_trends_plus_bruun_ssp585/mipNC_1980-2023_CoastSat_trends_plus_bruun_ssp585_SV.csv'),
    'CoSMoS-COAST-DSAS_trends': os.path.join(submissions_path, 'CoSMoS-COAST_SV/NC/DSAS_trends/mipNC_1980-2023_DSAS_SV.csv'),
    'CoSMoS-COAST-persistence': os.path.join(submissions_path, 'CoSMoS-COAST_SV/NC/persistence/mipNC_1980-2023_persistence_SV.csv'),
    'CRCHI': os.path.join(submissions_path, 'CRCHI/mipNC_1980-2023_waves_simplified_smoothed_shorelines-CRCHI.csv'),
    'GEOOCEAN-cross_eps': os.path.join(submissions_path, 'GEOOCEAN/mipNC_1980-2023_GeoOcean_cross_eps_bulk_SLP_Qs_partitions_N.csv'),
    'ICM-TCN': os.path.join(submissions_path, 'ICM-TCN/mipNC_1980-2023_ICM-TCN_eof.csv'),
    'IdSOR': os.path.join(submissions_path, 'IdSOR/mipNC_1980-2023_WaveBulkParameters_IdSOR.csv'),
    'mtraboulsi689': os.path.join(submissions_path, 'mtraboulsi689/mipNC_1980-2023_waves_simplified-MohammadTraboulsi.csv'),
    'UNSW-WRL-Mao': os.path.join(submissions_path, 'UNSW-WRL-Mao/shoreshop3NC/mipNC_1980-2023_GraphShore.csv'),
}